# 04c · Spark MLlib model: Decision Tree

**Task.** Binary classification: predict whether a driver files a claim (`OUTCOME = 1`).

**Why this model.** It is one of the models from Phase 2, so there is a scikit-learn result to compare against. It splits drivers with a sequence of yes/no questions about their features (for example, whether they are a beginner or own their vehicle), so each prediction of whether a driver will file a claim can be traced and explained. A single tree overfits when it grows deep, so its depth and the minimum number of drivers per node are tuned on the validation split. 

**Inputs.** `workspace.car_insurance.claims_gold_features` and `workspace.car_insurance.claims_model_split`.

**Outputs.** Fitted pipeline saved under `/Volumes/workspace/car_insurance/models/decision_tree`.

**Assumptions.** 
- Notebooks 01 to 03b have been run. 
- Serverless environment version 4 or later.
- Every column of the Gold feature table other than `ID` and `OUTCOME` is a model input. Text columns are treated as categories and numeric columns as numbers.

In [0]:
%run ./00_setup

# 00 · Shared setup: storage, table names and Delta helpers

**Purpose.** One place for everything the other notebooks share: where the data lives, what each Delta table is called, how tables are written and verified, and how models are split and scored. Keeping it here means all team members write to the same tables in the same way.

**How it is used.** This notebook is not run on its own. Every other notebook starts with a cell containing only `%run ./00_setup`, which runs these cells and makes the names and functions below available.

**Inputs.** None.

**Outputs.** No tables. It creates the schema `workspace.car_insurance` and the volumes `raw` (source files) and `models` (saved Spark ML models) if they do not exist yet.

**Assumptions**

- The catalog is named `workspace` (the Databricks Free Edition default). If yours differs, change `CATALOG` below; nothing else needs editing.
- Serverless compute.

**What it provides**

| Name | Use |
|---|---|
| `BRONZE_TABLE`, `SILVER_TABLE`, `GOLD_FEATURES_TABLE`, `GOLD_SUMMARY_TABLE`, `SPLIT_TABLE` | Full table names. Always use these instead of typing a table name |
| `CLAIMS_CSV`, `SPLIT_CSV`, `MODEL_DIR` | Paths inside the volumes |
| `require_source_files(...)` | Stops with a clear message if an uploaded file is missing |
| `write_delta(df, table_name)` | Saves a DataFrame as a Delta table |
| `validate_tables({...})` | Reads tables back and checks format, rows, columns and key |
| `load_model_splits()` | Train, validation and test DataFrames with the Phase 2 split |
| `evaluate_classifier(predictions)` | Accuracy, precision, recall, F1, ROC-AUC and confusion counts |

Setup loaded: tables in workspace.car_insurance, files in /Volumes/workspace/car_insurance


## 1. Data and split

`load_model_splits()` returns the same train (6,400), validation (1,600) and test (2,000) drivers as Phase 2. Fit on train, tune on validation, and score test once.

In [0]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler

NUMERIC_TYPES = ("tinyint", "smallint", "int", "bigint", "float", "double", "decimal", "boolean")
ALWAYS_CATEGORICAL = {"POSTAL_CODE"}

gold_types = dict(spark.table(GOLD_FEATURES_TABLE).drop("split").dtypes)
candidates = [c for c in gold_types if c not in (ID_COLUMN, LABEL_COLUMN) and not c.startswith("_")]

CATEGORICAL_FEATURES = [c for c in candidates if gold_types[c] == "string" or c in ALWAYS_CATEGORICAL]
NUMERIC_FEATURES = [c for c in candidates
                    if c not in CATEGORICAL_FEATURES and gold_types[c].startswith
                    (NUMERIC_TYPES)]
print(f"Categorical features ({len(CATEGORICAL_FEATURES)}):", CATEGORICAL_FEATURES)
print(f"Numeric features ({len(NUMERIC_FEATURES)}):", NUMERIC_FEATURES)

def prepare(df):
    """Type alignment only: label and numbers as double, categories as text."""
    return df.select(
        ID_COLUMN,
        F.col(LABEL_COLUMN).cast("double").alias("label"),
        *[F.col(c).cast("string").alias(c) for c in CATEGORICAL_FEATURES],
        *[F.col(c).cast("double").alias(c) for c in NUMERIC_FEATURES],
    )

train_df, validation_df, test_df = (prepare(split) for split in load_model_splits())

split_stats = [(name, split.agg(F.count("*").alias("drivers"), F.sum("label").alias("claims")).first())
               for name, split in zip(SPLIT_NAMES, (train_df, validation_df, test_df))]
display(spark.createDataFrame(
    [(name, int(r["drivers"]), int(r["claims"]), round(r["claims"] / r["drivers"], 4)) for name, r in split_stats],
    "split string, drivers long, claims long, claim_rate double"))
N_TRAIN = int(split_stats[0][1]["drivers"])

Categorical features (10): ['AGE', 'GENDER', 'RACE', 'DRIVING_EXPERIENCE', 'EDUCATION', 'INCOME', 'VEHICLE_YEAR', 'POSTAL_CODE', 'VEHICLE_TYPE', 'CAR_AGE']
Numeric features (10): ['CREDIT_SCORE', 'VEHICLE_OWNERSHIP', 'MARRIED', 'CHILDREN', 'ANNUAL_MILEAGE', 'SPEEDING_VIOLATIONS', 'DUIS', 'PAST_ACCIDENTS', 'INSURANCE_RISK', 'FAMILY']


split,drivers,claims,claim_rate
train,6400,1982,0.3097
validation,1600,495,0.3094
test,2000,656,0.328


## 2. Pipeline and training

| Stage | What it does | Fitted on train |
|---|---|---|
| `StringIndexer` | Maps each category to an index. The index carries category metadata, so the tree splits on groups of categories rather than treating the index as a number | The list of labels |
| `Imputer` (median) | Fills missing `CREDIT_SCORE` and `ANNUAL_MILEAGE` | The medians |
| `VectorAssembler` | Combines all inputs into one feature vector | Nothing |
| `DecisionTreeClassifier` | One tree, split by Gini impurity, with at most 32 candidate split points per feature (`maxBins`) | The splits |

Trees need no one-hot encoding or scaling, because a split depends only on the order of values. Every fitted step learns from the training drivers only, so no validation or test information reaches the model.

**Tuning.** Phase 2 tuned the tree with 5-fold cross-validation and chose a maximum depth of 7 with `min_samples_split = 100` and `min_samples_leaf = 1`. Spark has no `min_samples_split`; its closest setting is `minInstancesPerNode`, the minimum number of training drivers in each child of a split. The tuning cell below fits every combination of depth 3, 5, 8 or 10 and 25, 10 or 1 drivers per node on train, and keeps the one with the highest validation ROC-AUC. A tie goes to the simpler tree (shallower, then more drivers per node).

**Scoring.** For a single tree, Spark's `rawPrediction` holds the class counts of the leaf, which do not rank drivers by their claim probability. `evaluate_dtree` therefore scores ROC-AUC from the `probability` column.

In [0]:
SEED = 123

def build_pipeline(max_depth, min_instances):
    indexed = [f"{c}_index" for c in CATEGORICAL_FEATURES]
    imputed = [f"{c}_imputed" for c in NUMERIC_FEATURES]

    return Pipeline(stages=[
        StringIndexer(inputCols=CATEGORICAL_FEATURES, outputCols=indexed,
                      stringOrderType="alphabetDesc", handleInvalid="error"),
        Imputer(inputCols=NUMERIC_FEATURES, outputCols=imputed, strategy="median"),
        VectorAssembler(inputCols=indexed + imputed, outputCol="features"),
        DecisionTreeClassifier(featuresCol="features", labelCol="label", impurity="gini",
                               maxDepth=max_depth, minInstancesPerNode=min_instances,
                               maxBins=32, seed=SEED)
    ])

def evaluate_dtree(predictions):
    # A single tree's rawPrediction holds leaf class counts, not a ranking score; use the class probabilities.
    score_input = predictions.withColumn("rawPrediction", F.col("probability"))
    return evaluate_classifier(score_input, "label")

In [0]:
MAX_DEPTH_GRID = [3, 5, 8, 10]
MIN_INSTANCES_GRID = [25, 10, 1]

tuning_rows, best = [], None
for max_depth in MAX_DEPTH_GRID:
    for min_instances in MIN_INSTANCES_GRID:
        model = build_pipeline(max_depth, min_instances).fit(train_df)
        train_metrics = evaluate_dtree(model.transform(train_df))
        validation_metrics =evaluate_dtree(model.transform(validation_df))
        tuning_rows.append((max_depth, min_instances,
                            round(train_metrics["accuracy"], 4),
                            round(train_metrics["roc_auc"], 4),
                            round(validation_metrics["accuracy"], 4),
                            round(validation_metrics["roc_auc"], 4)))
        if best is None or validation_metrics["roc_auc"] > best["validation_roc_auc"]:
            best = {"maxDepth": max_depth, "minInstancesPerNode": min_instances,
                    "model": model, "validation_roc_auc": validation_metrics["roc_auc"]}
        del model

display(spark.createDataFrame(
    tuning_rows, "maxDepth int, minInstancesPerNode int, train_accuracy double, train_roc_auc double, "
    "validation_accuracy double, validation_roc_auc double"
))

best_model = best["model"]
print(f"Selected maxDepth={best['maxDepth']}, minInstancesPerNode={best['minInstancesPerNode']}: "
      f"validation ROC-AUC {best['validation_roc_auc']:.4f}")

maxDepth,minInstancesPerNode,train_accuracy,train_roc_auc,validation_accuracy,validation_roc_auc
3,25,0.8183,0.7803,0.8225,0.7817
3,10,0.8183,0.7803,0.8225,0.7817
3,1,0.8183,0.7803,0.8225,0.7817
5,25,0.8488,0.8837,0.8525,0.878
5,10,0.8488,0.8837,0.8525,0.878
5,1,0.8498,0.8868,0.8569,0.8757
8,25,0.8608,0.8885,0.8425,0.873
8,10,0.8648,0.9175,0.825,0.8936
8,1,0.8728,0.9361,0.8219,0.8808
10,25,0.863,0.9118,0.8381,0.8944


Selected maxDepth=10, minInstancesPerNode=10: validation ROC-AUC 0.8956


## 3. Evaluation and comparison with Phase 2

`evaluate_classifier` returns accuracy, precision, recall, F1 and ROC-AUC for the claim class plus the confusion-matrix counts, computed the same way for every model.

In [0]:
test_predictions = best_model.transform(test_df)
test_metrics = evaluate_dtree(test_predictions)

display(spark.createDataFrame(
    [(m, round(test_metrics[m], 4)) for m in ("accuracy", "precision", "recall", "f1", "roc_auc")],
    "metric string, test_value double"))

display(spark.createDataFrame(
    [("Actual: no claim", test_metrics["tn"], test_metrics["fp"]),
     ("Actual: claim", test_metrics["fn"], test_metrics["tp"])],
    "actual string, predicted_no_claim long, predicted_claim long"))

metric,test_value
accuracy,0.8315
precision,0.7602
recall,0.7104
f1,0.7344
roc_auc,0.9042


actual,predicted_no_claim,predicted_claim
Actual: no claim,1197,147
Actual: claim,190,466


In [0]:
METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc"]
PHASE2_DT = {"accuracy": 0.8570, "precision": 0.7790, "recall": 0.7590, "f1": 0.7690, "roc_auc": 0.911}

test_total = sum(test_metrics[k] for k in ("tp", "fp", "tn", "fn"))
baseline = {"accuracy": (test_metrics["tn"] + test_metrics["fp"]) / test_total,
            "precision": 0.0, "recall": 0.0, "f1": 0.0, "roc_auc": 0.5}

display(spark.createDataFrame(
    [(name, *[round(float(values[m]), 4) for m in METRICS])
     for name, values in [("Always 'no claim' (this test set)", baseline),
                          ("Phase 2 scikit-learn Decistion Tree (tuned)", PHASE2_DT),
                          ("Phase 3 Spark MLlib Decision Tree", test_metrics)]],
    "model string, accuracy double, precision double, recall double, f1 double, roc_auc double"))

display(spark.createDataFrame(
    [(m, round(test_metrics[m] - PHASE2_DT[m], 4)) for m in METRICS],
    "metric string, spark_minus_phase2 double"))

model,accuracy,precision,recall,f1,roc_auc
Always 'no claim' (this test set),0.672,0.0,0.0,0.0,0.5
Phase 2 scikit-learn Decistion Tree (tuned),0.857,0.779,0.759,0.769,0.911
Phase 3 Spark MLlib Decision Tree,0.8315,0.7602,0.7104,0.7344,0.9042


metric,spark_minus_phase2
accuracy,-0.0255
precision,-0.0188
recall,-0.0486
f1,-0.0346
roc_auc,-0.0068


### Why the numbers can differ

Both versions are tested on the same 2,000 drivers with the same target and metrics, so differences come from how the tree is built. One percentage point of accuracy is 20 test drivers.

| Aspect | Phase 2 (scikit-learn) | Phase 3 (Spark MLlib) |
|---|---|---|
| Training drivers | 8,000 (train + validation) | 6,400 (train) |
| Tuning | 5-fold cross-validation on ROC-AUC; chose depth 7, `min_samples_split = 100` | One validation split on ROC-AUC; depth 3, 5, 8, 10 × `minInstancesPerNode` 25, 10, 1 |
| Size limit | `min_samples_split`: a node needs 100 drivers to be split | No equivalent; `minInstancesPerNode` sets the minimum drivers in each child |
| Categories | One-hot: each split tests one category | Indexed: one split can group several categories |
| Continuous split points | Every distinct value | At most 32 bins per feature (`maxBins`) |
| Missing values | Linear-regression imputation | Training median; `CAR_AGE = unknown` for the 957 drivers with missing mileage |
| `INSURANCE_RISK` | Counts scaled to 0–1 before adding | Plain sum of the counts |

The largest expected effects are the tuning differences, because a single tree is sensitive to its depth and size limits, and the 1,600 fewer training drivers. Selecting by validation ROC-AUC favours deeper trees, which rank drivers well but can overfit their yes/no predictions: in the tuning table the depth-5 trees have the best validation accuracy, while ROC-AUC selects a deeper tree. Phase 2's own decision tree varied by about one percentage point between runs and feature sets (report Table V: 85.70% accuracy, 0.911 ROC-AUC), so gaps of that size should not be over-interpreted.

## 4. Save and reload

In [0]:
MODEL_PATH = f"{MODEL_DIR}/decision_tree"
best_model.write().overwrite().save(MODEL_PATH)
reloaded = PipelineModel.load(MODEL_PATH)

reloaded_metrics = evaluate_dtree(PipelineModel.load(MODEL_PATH).transform(test_df))
for name in ("tp", "fp", "tn", "fn"):
    assert reloaded_metrics[name] == test_metrics[name], "The reloaded model predicts differently"
print(f"Reloaded model reproduces the test results: accuracy {reloaded_metrics['accuracy']:.4f}, "
      f"ROC-AUC {reloaded_metrics['roc_auc']:.4f}")


Reloaded model reproduces the test results: accuracy 0.8315, ROC-AUC 0.9042
